In [ ]:
import io
import requests
import pandas as pd

GDC_BASE = "https://api.gdc.cancer.gov"

In [ ]:
file_filter = {
    "op": "and",
    "content": [
        {"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}},
        {"op": "in", "content": {"field": "files.data_type", "value": ["Pathology Report"]}},
        {"op": "in", "content": {"field": "files.data_format", "value": ["pdf"]}},
        {"op": "in", "content": {"field": "files.access", "value": ["open"]}},
    ],
}
resp = requests.post(
    f"{GDC_BASE}/files",
    json={
        "filters": file_filter,
        "fields": ",".join(["file_name", "cases.submitter_id", "cases.project.project_id", "cases.primary_site", "experimental_strategy", "data_format", "md5sum"]),
        "format": "TSV",
        "size": "100000",
    },
)
resp.raise_for_status()
df = pd.read_csv(io.StringIO(resp.text), sep="\t")

In [ ]:
df

In [ ]:
df.rename(
    columns={
        "cases.0.primary_site": "primary_site",
        "cases.0.project.project_id": "project_id",
        "cases.0.submitter_id": "case_id",
    }
)[["file_name", "case_id", "project_id", "primary_site"]].sort_values(
    ["file_name"]
).to_csv(
    "../metadata/tcga-report-metadata.csv", index=False
)

In [ ]:
resp = requests.post(
    f"{GDC_BASE}/files",
    json={
        "filters": file_filter,
        "return_type": "manifest",
    },
)
resp.raise_for_status()

In [ ]:
with open("../manifests/tcga-report.txt", "w") as f:
    f.write(resp.text)